# SingVox SPOTER V3 — 95-Class Training on 100-Output Architecture (GPU)

This notebook implements the official training and evaluation pipeline for **SignVox SPOTER V3** on the canonical **100-Class ASL Vocabulary** specification (95 supported classes with active training data; 5 reserved slots).

---  
### ⚠️ LEGAL & COMPLIANCE NOTICE (MICROSOFT RESEARCH LICENSE)
The Microsoft Research License for ASL Citizen strictly prohibits distributing, sharing, or transferring dataset materials, modifications (including extracted landmark arrays), or signer-identifiable personal data to third-party public cloud platforms without explicit authorization (§1(c), §2(e), §3).  

**DO NOT upload data to a public Google Drive or public Colab runtime without an authorized agreement.**  
This notebook is authorized for execution exclusively in:
1. Google Colab connected to a **Local Runtime / Private On-Premise GPU Gateway**, OR
2. An authorized private cloud environment with an active Data Protection Agreement (DPA) and ephemeral storage destruction policy upon research completion.


In [ ]:
# Step 1: Environment Check & Dependencies
!pip install -q torch numpy matplotlib scikit-learn

import os
import sys
import json
import csv
import time
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Step 2: Seed Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print('Reproducibility seed set to 42.')


In [ ]:
# Step 3: SPOTER V3 Architecture (100 Classes, 366-D)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.pe = nn.Parameter(torch.empty(1, max_len, d_model))
        nn.init.uniform_(self.pe, -0.1, 0.1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class SpoterV3(nn.Module):
    def __init__(
        self,
        num_classes: int = 100,
        input_dim: int = 366,
        hidden_dim: int = 512,
        nhead: int = 8,
        num_encoder_layers: int = 6,
        num_decoder_layers: int = 6,
        dim_feedforward: int = 2048,
        dropout: float = 0.1
    ):
        super().__init__()
        self.num_classes = num_classes
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.pos_encoder = PositionalEncoding(hidden_dim, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        self.class_query = nn.Parameter(torch.rand(1, 1, hidden_dim))
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        pad_mask = ~attention_mask if attention_mask is not None else None
        memory = self.transformer_encoder(x, src_key_padding_mask=pad_mask)
        query = self.class_query.expand(x.size(0), -1, -1)
        out = self.transformer_decoder(tgt=query, memory=memory, memory_key_padding_mask=pad_mask)
        return self.classifier(out.squeeze(1))


In [ ]:
# Step 4: Normalization (Verified GroupNorm Scalars on Full 3,439 Dataset)
VERIFIED_SCALARS = {
    'left_hand': 0.3937907814979553,
    'right_hand': 0.36684879660606384,
    'pose_xyz': 1.8130728006362915,
    'face': 0.31350088119506836
}

def get_feature_channel_indices():
    lh = list(range(0, 63))
    rh = list(range(63, 126))
    pose = [126 + i*5 + j for i in range(33) for j in range(3)]
    face = list(range(291, 366))
    return lh, rh, pose, face

def apply_groupnorm(seqs, scalars=VERIFIED_SCALARS):
    lh, rh, pose, face = get_feature_channel_indices()
    scaled = seqs.clone().float()
    scaled[:, :, lh] /= float(scalars.get('left_hand', 1.0))
    scaled[:, :, rh] /= float(scalars.get('right_hand', 1.0))
    scaled[:, :, pose] /= float(scalars.get('pose_xyz', 1.0))
    scaled[:, :, face] /= float(scalars.get('face', 1.0))
    return scaled


In [ ]:
# Step 5: Dataset & Dataloaders with Dynamic Sequence Padding
class SpoterV3Dataset(Dataset):
    def __init__(self, manifest_csv, features_dir, mapping_json, split='train'):
        with open(mapping_json, 'r', encoding='utf-8') as f:
            mapping = json.load(f)
        self.asl_to_idx = {k: v['class_index'] for k, v in mapping.items()}
        self.samples = []
        with open(manifest_csv, 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                if row.get('source_split') == split:
                    asl_id = row['asl_id']
                    if asl_id in self.asl_to_idx:
                        base = Path(row['video_file']).stem
                        npy = Path(features_dir) / f'{base}_features.npy'
                        if npy.exists():
                            self.samples.append({'npy': str(npy), 'idx': self.asl_to_idx[asl_id]})
        self.samples.sort(key=lambda s: s['npy'])
        print(f'Loaded {len(self.samples)} {split} samples.')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        seq = np.load(self.samples[i]['npy'])
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.samples[i]['idx'], dtype=torch.long)

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0.0)
    mask = torch.zeros((len(batch), padded.size(1)), dtype=torch.bool)
    for i, l in enumerate(lengths):
        mask[i, :l] = True
    return padded, mask, torch.stack(labels)


In [ ]:
# Step 6: Training Pipeline with Early Stopping & Checkpointing
def train_spoter_v3(
    manifest_path,
    features_dir,
    mapping_path,
    epochs=50,
    batch_size=32,
    lr=1e-4,
    patience=10,
    checkpoint_save_dir='./checkpoints_v3',
    resume_path=None
):
    os.makedirs(checkpoint_save_dir, exist_ok=True)
    train_ds = SpoterV3Dataset(manifest_path, features_dir, mapping_path, 'train')
    val_ds = SpoterV3Dataset(manifest_path, features_dir, mapping_path, 'val')
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model = SpoterV3(num_classes=100).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    start_epoch = 0
    best_val_loss = float('inf')
    best_val_acc = 0.0
    early_stop_counter = 0

    if resume_path and os.path.exists(resume_path):
        cp = torch.load(resume_path, map_location=device)
        model.load_state_dict(cp['model_state_dict'])
        optimizer.load_state_dict(cp['optimizer_state_dict'])
        scheduler.load_state_dict(cp['scheduler_state_dict'])
        start_epoch = cp['epoch'] + 1
        best_val_loss = cp.get('best_val_loss', float('inf'))
        best_val_acc = cp.get('best_val_acc', 0.0)
        print(f'Resumed from epoch {start_epoch} (Best Val Acc: {best_val_acc*100:.2f}%)')

    for ep in range(start_epoch, epochs):
        t0 = time.time()
        model.train()
        t_loss, correct, total = 0.0, 0, 0
        for seqs, mask, targets in train_loader:
            seqs, mask, targets = seqs.to(device), mask.to(device), targets.to(device)
            scaled = apply_groupnorm(seqs).to(device)
            optimizer.zero_grad()
            out = model(scaled, attention_mask=mask)
            loss = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item() * targets.size(0)
            correct += (out.argmax(-1) == targets).sum().item()
            total += targets.size(0)
        scheduler.step()

        # Validation
        model.eval()
        v_loss, v_correct, v_top3, v_total = 0.0, 0, 0, 0
        with torch.no_grad():
            for seqs, mask, targets in val_loader:
                seqs, mask, targets = seqs.to(device), mask.to(device), targets.to(device)
                scaled = apply_groupnorm(seqs).to(device)
                out = model(scaled, attention_mask=mask)
                loss = criterion(out, targets)
                v_loss += loss.item() * targets.size(0)
                preds = out.argmax(-1)
                v_correct += (preds == targets).sum().item()
                top3 = torch.topk(out, k=3, dim=-1).indices
                v_top3 += (top3 == targets.unsqueeze(-1)).any(dim=-1).sum().item()
                v_total += targets.size(0)

        train_acc = correct / max(total, 1)
        val_acc = v_correct / max(v_total, 1)
        val_top3_acc = v_top3 / max(v_total, 1)
        avg_val_loss = v_loss / max(v_total, 1)
        elapsed = time.time() - t0

        print(f'Epoch {ep+1:02d}/{epochs:02d} [{elapsed:.1f}s] | Train Acc: {train_acc*100:.2f}% | Val Loss: {avg_val_loss:.4f} Top-1: {val_acc*100:.2f}% Top-3: {val_top3_acc*100:.2f}%')

        cp_payload = {
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scalars': VERIFIED_SCALARS,
            'val_loss': avg_val_loss,
            'val_acc': val_acc,
            'best_val_loss': best_val_loss,
            'best_val_acc': best_val_acc
        }
        torch.save(cp_payload, os.path.join(checkpoint_save_dir, 'spoter_v3_latest.pt'))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(cp_payload, os.path.join(checkpoint_save_dir, 'spoter_v3_best_val_acc.pt'))
            print(f'  --> Saved new best val acc checkpoint ({best_val_acc*100:.2f}%)')

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(cp_payload, os.path.join(checkpoint_save_dir, 'spoter_v3_best_val_loss.pt'))
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if patience > 0 and early_stop_counter >= patience:
                print(f'  [Early Stop] Patience ({patience}) reached at epoch {ep+1}. Stopping.')
                break

    print(f'Training finished. Best Val Acc: {best_val_acc*100:.2f}% | Best Val Loss: {best_val_loss:.4f}')


In [ ]:
# Step 7: Final Evaluation on Held-Out Test Split (1,379 Samples)
def evaluate_test_set(checkpoint_path, manifest_path, features_dir, mapping_path):
    print(f'Evaluating held-out test split using checkpoint: {checkpoint_path}')
    test_ds = SpoterV3Dataset(manifest_path, features_dir, mapping_path, 'test')
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

    model = SpoterV3(num_classes=100).to(device)
    cp = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(cp['model_state_dict'])
    model.eval()

    correct, top3_correct, total = 0, 0, 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for seqs, mask, targets in test_loader:
            seqs, mask, targets = seqs.to(device), mask.to(device), targets.to(device)
            scaled = apply_groupnorm(seqs).to(device)
            out = model(scaled, attention_mask=mask)
            preds = out.argmax(-1)
            top3 = torch.topk(out, k=3, dim=-1).indices
            correct += (preds == targets).sum().item()
            top3_correct += (top3 == targets.unsqueeze(-1)).any(dim=-1).sum().item()
            total += targets.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(targets.cpu().tolist())

    top1_acc = correct / max(total, 1)
    top3_acc = top3_correct / max(total, 1)
    print('=' * 60)
    print(f'Held-Out Test Results ({total} samples):')
    print(f'  Top-1 Test Accuracy: {top1_acc*100:.2f}%')
    print(f'  Top-3 Test Accuracy: {top3_acc*100:.2f}%')
    print('=' * 60)
